### 🎓 02 · 枚举、联合类型与控制流收窄 (`enum` / `Union`)
在复杂状态机与多态输入处理中，避免硬编码魔术数字是高质量 TypeScript 工程的核心：
- **`enum` 枚举**：编译期提供具名类型约束，运行期自动展开为双向映射字典对象。
- **联合类型 (`number | string`)**：配合 `typeof` 条件守卫，让编译器在各分支内自动收窄具体类型。

In [ ]:
// 1. 定义任务状态枚举与联合类型输入解析函数
enum JobStatus {
  Queued = 0,
  Running = 1,
  Completed = 2,
  Failed = 3
}

type MetricInput = number | string;

function normalizeWeight(raw: MetricInput): number {
  if (typeof raw === "number") {
    return raw;
  }
  return raw.length * 10;
}

const combinedWeight: number = normalizeWeight(60) + normalizeWeight("high");
console.log("收窄计算权重:", combinedWeight, "初始枚举值:", JobStatus.Queued);

In [ ]:
// 2. 编写确定性状态转移函数 stepStatus
function stepStatus(cur: JobStatus, ok: boolean): JobStatus {
  if (cur === JobStatus.Queued) {
    return JobStatus.Running;
  }
  if (cur === JobStatus.Running) {
    return ok ? JobStatus.Completed : JobStatus.Failed;
  }
  return cur;
}

let stDemo: JobStatus = JobStatus.Queued;
stDemo = stepStatus(stDemo, true);
stDemo = stepStatus(stDemo, true);
console.log("状态流转终点:", stDemo, "名称:", JobStatus[stDemo]);

In [ ]:
// 3. 点击右上角 🐞 单步调试：批量驱动状态机并在右侧 WATCH 观察 Map 哈希统计桶
const signals: boolean[] = [true, false, true, true, false];
const stats = new Map<string, number>();
stats.set("Completed", 0);
stats.set("Failed", 0);

for (let i: number = 0; i < signals.length; i++) {
  let st: JobStatus = JobStatus.Queued;
  st = stepStatus(st, true);
  st = stepStatus(st, signals[i]);
  const key: string = st === JobStatus.Completed ? "Completed" : "Failed";
  const prev: number = stats.get(key) || 0;
  stats.set(key, prev + 1);
  checkpoint(i, key, stats.get(key));
}

console.log("成功任务数:", stats.get("Completed"), "失败任务数:", stats.get("Failed"));